## Rag using Web based loader

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' 

In [6]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

In [14]:
MODEL_NAME = "gpt-5.1"
EMBEDDING_MODEL = "text-embedding-3-small"
VECTOR_STORE_NAME = "medical-rag"
PERSIST_DIRECTORY = "./medical-rag-db"

In [15]:
llm = ChatOpenAI(
    api_key = API_KEY,
    base_url= BASE_URL, 
    model = MODEL_NAME
).with_config(
    run_name= MODEL_NAME
)

embedding_model = OpenAIEmbeddings(
    api_key = API_KEY,
    base_url = BASE_URL,
    model = EMBEDDING_MODEL
)

In [23]:
medical_info_urls = [
    "https://www.who.int/health-topics/",
    "https://www.mayoclinic.org/diseases-conditions",
    "https://medlineplus.gov/symptoms.html",
    "https://www.webmd.com/a-to-z-guides/diseases-conditions",
]

In [24]:
docs = []
for i, url in enumerate(medical_info_urls):
    try:
        loader = WebBaseLoader(url)
        docs.extend(loader.load())
    except Exception as e:
        print(f'Faieled loading the doc from {url}')
        print(e)
        continue

print(len(docs))

4


In [25]:
print(docs[0])
print(docs[0].model_dump().keys())

page_content='      
	Health topics
                     
   Skip to main content       


 







Global


Regions







WHO Regional websites







Africa





Americas





South-East Asia





Europe





Eastern Mediterranean





Western Pacific









   













When autocomplete results are available use up and down arrows to review and enter to select.
















        Select language
    

Select language
English
العربية
中文
Français
Русский
Español
Português




        
            










       











Home













Health Topics








All topicsABCDEFGHIJKLMNOPQRSTUVWXYZ







Resources


Fact sheets


Facts in pictures


Multimedia


Podcasts


Publications


Questions and answers


Tools and toolkits










Popular


Dengue


Endometriosis


Excessive heat


Herpes


Mental disorders


Mpox

















Countries








All countriesABCDEFGHIJKLMNOPQRSTUVWXYZ







Regions


Africa


Americas


Europe


Eastern Mediterranean


So

In [28]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size =  500,
    chunk_overlap = 40
)
doc_split = text_splitter.split_documents(docs)
len(doc_split)

54

In [29]:
if os.path.exists(PERSIST_DIRECTORY):
    vector_store = Chroma(
        persist_directory = PERSIST_DIRECTORY,
        embedding_function = embedding_model,
        collection_name = VECTOR_STORE_NAME
    )
    existing_count = len(vector_store.get()['ids'])
    new_count = len(doc_split)

    if existing_count != new_count:
        vector_store.delete_collection()
        vector_store = Chroma(
            persist_directory = PERSIST_DIRECTORY,
            embedding_function = embedding_model,
            collection_name = VECTOR_STORE_NAME
        )
        vector_store.add_documents(doc_split)
else:
    vector_store = Chroma(
        persist_directory = PERSIST_DIRECTORY,
        embedding_function = embedding_model,
        collection_name = VECTOR_STORE_NAME
    )
    vector_store.add_documents(doc_split)


In [31]:
from langchain.tools import tool

@tool('ReteriverTool', response_format = 'content_and_artifact')
def reterieve_context(query: str, num_docs: int = 3):
    '''Tererive information to help answer a query'''
    reterived_docs = vector_store.similarity_search(query, k=num_docs)
    serialized_docs = '\n\n'.join((f'Source: {doc.metadata['source']}\nContent: {doc.page_content}') for doc in reterived_docs)
    return serialized_docs, reterived_docs

Creating the agent

In [37]:
from langchain.agents import create_agent

tools = [reterieve_context]
prompt = ('''
You have access to a tool that retrieves context from a web pages.
Use the tool to help answer user queries and keep the answer short.
''')

agent = create_agent(llm, tools, system_prompt=prompt)

In [38]:
query = "I have the flu. What are some recommended at-home remedies?"

In [39]:
for event in agent.stream(
    {'messages': [{'role': 'user', 'content': query}]},
    stream_mode = 'values'
):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

I have the flu. What are some recommended at-home remedies?
================================== Ai Message ==================================
Tool Calls:
  ReteriverTool (call_OQKeUoOHfYn4AfPThJy8QH9d)
 Call ID: call_OQKeUoOHfYn4AfPThJy8QH9d
  Args:
    query: at home remedies for influenza
    num_docs: 3
================================= Tool Message =================================
Name: ReteriverTool

Source: https://www.who.int/health-topics/
Content: Diseases and conditions

Hypertension







Health interventions

In vitro diagnostics







Health interventions

Infant nutrition







Health systems

Infection prevention and control







Conditions

Infertility







Diseases and conditions

Influenza (avian and other zoonotic)







Diseases and conditions

Influenza (seasonal)







Socio-political determinants

Infodemic







Socio-political determinants

Intellectual property and trad

In [47]:
resp = agent.invoke({'messages': 'What are the early signs of diabetes?'})

In [51]:
len(resp['messages'])

4

In [53]:
resp['messages'][-1].content

'Common early signs of diabetes (especially type 2) include:\n\n1. **Increased thirst and frequent urination**  \n   High blood sugar pulls fluid from tissues, making you thirsty and causing you to pee more, especially at night.\n\n2. **Increased hunger**  \n   Your cells aren’t getting enough glucose, so you may feel hungry even after eating.\n\n3. **Unexplained weight loss**  \n   More common in type 1, but can happen in type 2. Your body starts breaking down fat and muscle for energy.\n\n4. **Fatigue**  \n   Feeling unusually tired or weak because your cells can’t use glucose properly.\n\n5. **Blurred vision**  \n   High blood sugar can cause fluid shifts that swell the lenses in your eyes.\n\n6. **Slow-healing sores or frequent infections**  \n   High sugar impairs circulation and the immune system, so cuts, sores, or infections (skin, gums, vaginal, urinary) may be more frequent or slow to heal.\n\n7. **Darkened skin in body folds (acanthosis nigricans)**  \n   Velvety, darker pat